In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv('MNIST.csv')
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
data = np.array(data)
print(data.shape)
data = data.T
data.shape

(42000, 785)


(785, 42000)

In [4]:
row, col = data.shape

data = data[:, np.random.permutation(col)]

# split 80/20
train, test = data[:, :int(col * 0.8)], data[:, int(col * 0.8):]

In [5]:
x_train = train[1::] / 255.
y_train = train[0]

x_test = test[1::] / 255.
y_test = test[0]

In [6]:
def init_params():
    w1 = np.random.rand(10, 784) - 0.5
    b1 = np.zeros((10, 1))
    w2 = np.random.rand(10, 10) - 0.5
    b2 = np.zeros((10, 1))
    return w1, b1, w2, b2

def relu(z):
    return np.maximum(0, z)

def softmax(z):
    e = np.exp(z - np.max(z, axis=0, keepdims=True))
    return e / np.sum(e, axis=0, keepdims=True)

def forward(w1, b1, w2, b2, x):
    z1 = w1.dot(x) + b1
    a1 = relu(z1)
    z2 = w2.dot(a1) + b2
    a2 = softmax(z2)
    return z1, a1, z2, a2

def one_hot_encoder(y):
    return np.eye(10)[y].T

def drelu(z):
    return z > 0

def back(z1, a1, z2, a2, w2, x, y):
    y = one_hot_encoder(y)
    m = y.shape[1]

    dz2 = a2 - y
    dw2 = 1 / m * dz2.dot(a1.T)
    db2 = 1 / m * np.sum(dz2, axis=1, keepdims=True)
    dz1 = w2.T.dot(dz2) * drelu(z1)
    dw1 = 1 / m * dz1.dot(x.T)
    db1 = 1 / m * np.sum(dz1, axis=1, keepdims=True)
    return dw1, db1, dw2, db2

def update(w1, b1, w2, b2, dw1, db1, dw2, db2, alpha):
    w1 = w1 - alpha * dw1
    b1 = b1 - alpha * db1
    w2 = w2 - alpha * dw2
    b2 = b2 - alpha * db2
    return w1, b1, w2, b2

def get_pred(a2):
    return np.argmax(a2, 0)

def get_acc(predictions, y):
    return np.sum(predictions == y) / y.size

def train_model(x, y, iter, alpha):
    w1, b1, w2, b2 = init_params()
    for i in range(iter):
        z1, a1, z2, a2 = forward(w1, b1, w2, b2, x)
        dw1, db1, dw2, db2 = back(z1, a1, z2, a2, w2, x, y)
        w1, b1, w2, b2 = update(w1, b1, w2, b2, dw1, db1, dw2, db2, alpha)
        if i % 10 == 0:
            print("iteration:", i, "accuracy:", get_acc(get_pred(a2), y))
    return w1, b1, w2, b2

In [7]:
w1, b1, w2, b2 = train_model(x_train, y_train, 500, 0.1)

iteration: 0 accuracy: 0.10705357142857143
iteration: 10 accuracy: 0.1855952380952381
iteration: 20 accuracy: 0.3042261904761905
iteration: 30 accuracy: 0.40476190476190477
iteration: 40 accuracy: 0.4705654761904762
iteration: 50 accuracy: 0.5175297619047619
iteration: 60 accuracy: 0.5536309523809524
iteration: 70 accuracy: 0.5863095238095238
iteration: 80 accuracy: 0.6141666666666666
iteration: 90 accuracy: 0.6370833333333333
iteration: 100 accuracy: 0.6556547619047619
iteration: 110 accuracy: 0.671875
iteration: 120 accuracy: 0.6861607142857142
iteration: 130 accuracy: 0.699672619047619
iteration: 140 accuracy: 0.710952380952381
iteration: 150 accuracy: 0.7219047619047619
iteration: 160 accuracy: 0.7311904761904762
iteration: 170 accuracy: 0.7384226190476191
iteration: 180 accuracy: 0.7463095238095238
iteration: 190 accuracy: 0.7533630952380952
iteration: 200 accuracy: 0.7588690476190476
iteration: 210 accuracy: 0.7640178571428572
iteration: 220 accuracy: 0.7693154761904762
iteration

In [8]:
_, _, _, a2_test = forward(w1, b1, w2, b2, x_test)
test_pred = get_pred(a2_test)
print("test accuracy:", get_acc(test_pred, y_test))

test accuracy: 0.8473809523809523
